# 연습문제 2-7 ~ 2-8 풀이: 고수준 API로 이전 모델 다시 만들기

2-3절 연습문제 2-7 ~ 2-8의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 방법으로도 구현할 수 있다.

- 본문 예제 코드는 `code_examples/ch01/`, `code_examples/ch02/` 아래 예제 노트북을 참고한다.

In [1]:
# torch 라이브러리 임포트
import torch
import torch.nn as nn
import torch.optim as optim

## 연습문제 2-7

> 1장에서 외계 행성의 물리 법칙을 검증하려고 직접 구현한 회귀 분석 모델을,
> 파이토치 고수준 API로 다시 만들어 보자.
>
> 힌트: `nn.Linear`는 선형 계층으로 제곱을 직접 계산할 수 없다. 이를 어떻게 처리할 지 생각해 보자.

## 사전 준비

1-3절의 데이터 생성(1-3절 예제 노트북의 '가상의 관측 데이터 생성', [코드 1-30]의 훈련, 평가 데이터 분리)과 2-3절의 학습 루프([코드 2-16])를 가져온다.
학습 로그는 출력하지 않고 마지막 손실만 반환하는 조용한 학습 함수를 구현해 사용한다.
1장의 결과와 비교할 수 있도록 시드(8), 학습률(0.0001), 에포크(15), 초깃값(0.96, -0.56)을 1장과 같게 맞춘다.

In [2]:
SEED = 8
LR = 0.0001         # 학습률
EPOCHS = 15         # 전체 학습 에포크
SAMPLE_SIZE = 50    # 생성할 샘플의 수

torch.manual_seed(SEED)
X = torch.rand((SAMPLE_SIZE, 1)) * 10                     # 관측시간
Y_ideal = 4.9 * X ** 2                                    # 이상적인 낙하거리
Y_true = Y_ideal + torch.randn((SAMPLE_SIZE, 1)) * Y_ideal * 0.1    # 관측 오차 포함

TRAIN_SIZE = int(SAMPLE_SIZE * 0.8)
X_train, Y_train = X[:TRAIN_SIZE], Y_true[:TRAIN_SIZE]
X_test, Y_test = X[TRAIN_SIZE:], Y_true[TRAIN_SIZE:]
print(f'훈련 데이터 {X_train.shape[0]}개, 평가 데이터 {X_test.shape[0]}개')

훈련 데이터 40개, 평가 데이터 10개


In [3]:
def train_quiet(model, X, Y_true, criterion, optimizer, epochs):
    """[코드 2-16]과 같은 고수준 API 학습 루프. 마지막 손실만 반환한다."""
    model.train()                           # 학습 모드 선언
    for _ in range(epochs):
        loss = criterion(model(X), Y_true)  # 순전파와 손실 계산
        optimizer.zero_grad()               # 기울기 초기화
        loss.backward()                     # 역전파
        optimizer.step()                    # 파라미터 최적화
    return loss.item()

**y = ax^2 + b를 nn.Linear 하나로 표현하기**
- `nn.Linear`는 입력에 가중치를 곱하고 편향을 더하는 계층: 입력으로 x 대신 x^2을 넣으면 가중치가 a, 편향이 b
- 제곱은 모델의 `forward()` 안에서 처리
- 1장과 같은 초깃값(a = 0.96, b = -0.56)에서 시작해 결과를 비교
    - 초깃값은 `fill_(값)`으로 지정: 텐서의 모든 요소를 그 값으로 채움
    - 이름 끝의 `_`는 새 텐서를 만들지 않고 값만 바꾸는 제자리 연산 메서드(`zero_()`와 같은 규칙)이며, 파라미터는 `torch.no_grad()` 안에서 바꿈

In [4]:
class QuadraticRegression(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear_layer = nn.Linear(in_features=1, out_features=1)    # 가중치 a, 편향 b

    def forward(self, x):
        return self.linear_layer(x ** 2)            # 제곱은 forward()에서 처리

model = QuadraticRegression()
with torch.no_grad():                               # 1장([코드 1-33])과 같은 초깃값
    model.linear_layer.weight.fill_(0.96)
    model.linear_layer.bias.fill_(-0.56)
criterion = nn.MSELoss()
optimizer = optim.SGD(model.parameters(), lr=LR)
print(model)

QuadraticRegression(
  (linear_layer): Linear(in_features=1, out_features=1, bias=True)
)


**학습하고 1장의 결과와 비교하기**
- 훈련 데이터로 15 에포크 학습한 뒤 평가 데이터로 평가 손실을 계산
- 1장 직접 구현 결과: a = 5.12, b = -0.49, 훈련 손실 380.41, 평가 손실 805.22

In [5]:
train_loss = train_quiet(model, X_train, Y_train, criterion, optimizer, EPOCHS)
model.eval()                                        # 평가 모드 선언
with torch.no_grad():
    test_loss = criterion(model(X_test), Y_test).item()

a, b = model.linear_layer.weight.item(), model.linear_layer.bias.item()
print(f'고수준 API 결과: a = {a:.2f}, b = {b:.2f}')
print(f'    훈련 손실 {train_loss:.2f}, 평가 손실 {test_loss:.2f}')
print('1장 직접 구현:   a = 5.12, b = -0.49')
print('    훈련 손실 380.41, 평가 손실 805.22')

고수준 API 결과: a = 5.12, b = -0.49
    훈련 손실 380.40, 평가 손실 805.19
1장 직접 구현:   a = 5.12, b = -0.49
    훈련 손실 380.41, 평가 손실 805.22


**결과 확인: 1장의 직접 구현과 거의 같은 결과가 나온다**
- 같은 초깃값에서 시작하면 파라미터가 같고, 손실도 거의 같음(380.40 대 380.41, 805.19 대 805.22)
- 고수준 API는 다른 모델이 아니라 같은 모델과 같은 학습 방법을 간결하게 표현한 것
- 파라미터 선언, 기울기 계산 후 초기화, 최적화 식이 모두 `nn.Linear`, `optimizer.zero_grad()`, `optimizer.step()`으로 대체

**직접 해 보기: 미리 제곱한 데이터를 입력하면?**
- `nn.Linear`에 인자를 제곱해 전달하는 대신, 미리 데이터를 제곱해 입력
- forward()에서 제곱한 모델과 결과가 같을까?

In [6]:
linear = nn.Linear(1, 1)                            # 모델은 선형 계층 하나
with torch.no_grad():
    linear.weight.fill_(0.96)
    linear.bias.fill_(-0.56)
optimizer2 = optim.SGD(linear.parameters(), lr=LR)

# 미리 제곱한 데이터 텐서 생성, 지수 연산자(**)도 브로드캐스팅 대상 연산에 포함됨
X_square_train = X_train ** 2
X_square_test = X_test ** 2                 # 평가 데이터도 제곱한 값을 사용

train_loss2 = train_quiet(linear, X_square_train, Y_train, criterion, optimizer2, EPOCHS)
with torch.no_grad():
    test_loss2 = criterion(linear(X_square_test), Y_test).item()      
print(f'데이터에서 제곱: a = {linear.weight.item():.2f}, b = {linear.bias.item():.2f}')
print(f'    훈련 손실 {train_loss2:.2f}, 평가 손실 {test_loss2:.2f}')
print(f'forward()에서 제곱한 모델과 같은가: {abs(train_loss2 - train_loss) < 1e-3}')

데이터에서 제곱: a = 5.12, b = -0.49
    훈련 손실 380.40, 평가 손실 805.19
forward()에서 제곱한 모델과 같은가: True


**확인**
- 데이터에서 미리 제곱해도 파라미터와 손실이 같음: 제곱을 처리하는 위치만 다를 뿐 계산은 같은 모델
- 단, 평가 데이터에도 같은 제곱을 적용해야 함: 빠뜨리면 모델이 훈련 때와 다른 입력을 받게 되어 평가 손실이 맞지 않음
- 모델 안에서 입력을 제곱하면 데이터를 사용할 때 제곱을 빠뜨리는 실수를 피할 수 있어 안전함

## 연습문제 2-8

> 입력이 세 개인 [연습문제 2-6]의 AND 게이트 시뮬레이터 모델을 파이토치 고수준 API로 다시 만들어 보자.

**Perceptron 클래스로 입력 세 개 모델 만들기**
- 본문 [코드 2-10]의 `Perceptron(in_features, out_features)`는 입력 개수를 인자로 받으므로 `Perceptron(3, 1)`로 생성
- 데이터는 2-6과 같은 8개 샘플, 설정은 2-6의 본문 설정(학습률 0.1, 1,000 에포크)부터 시작

In [7]:
class Perceptron(nn.Module):                        # 본문 [코드 2-10]
    def __init__(self, in_features, out_features):
        super().__init__()
        self.linear_layer = nn.Linear(in_features, out_features)
        self.activation = nn.Sigmoid()

    def forward(self, x):
        return self.activation(self.linear_layer(x))

X = torch.tensor([[float(bit) for bit in f'{i:03b}'] for i in range(8)])   # (8, 3)
Y = torch.zeros((8, 1))
Y[7] = 1.                                          # (1, 1, 1)일 때만 1
model = Perceptron(3, 1)
print(model)

Perceptron(
  (linear_layer): Linear(in_features=3, out_features=1, bias=True)
  (activation): Sigmoid()
)


**학습하고 2-6과 같은 하이퍼파라미터 문제 확인하기**
- 2-6에서 실패한 본문 설정과 성공한 설정을 같은 시드로 비교
- 분류 결과(0.5 문턱값으로 8개 샘플을 모두 맞히는가)로 성공 여부 판단

In [8]:
def run_and3(epochs, lr, seed=2):
    torch.manual_seed(seed)                         # 초깃값을 재현할 수 있도록 고정
    model = Perceptron(3, 1)
    optimizer = optim.SGD(model.parameters(), lr=lr)
    loss = train_quiet(model, X, Y, nn.MSELoss(), optimizer, epochs)
    model.eval()
    with torch.no_grad():
        ok = torch.equal((model(X) >= 0.5).float(), Y)
    return loss, ok, model

print('학습률 에포크   손실 분류 성공')
for lr, epochs in ((0.1, 1000), (0.1, 2000), (0.3, 1000)):
    loss, ok, _ = run_and3(epochs, lr)
    print(f'{lr:6.1f} {epochs:6,d} {loss:6.4f} {str(ok):>9}')

학습률 에포크   손실 분류 성공
   0.1  1,000 0.0760     False
   0.1  2,000 0.0515      True
   0.3  1,000 0.0393      True


**결과 확인: 연습문제 2-6과 같은 분류 결과**
- 학습률 0.1, 1,000 에포크는 손실이 작아도 (1, 1, 1)을 잘못 분류, 에포크를 늘리거나 학습률을 키우면 성공
- 제대로 구현했다면 직접 구현이든 고수준 API든 동작 방식과 하이퍼파라미터가 학습에 미치는 영향은 동일함

**직접 구현과 가중치 형태 비교하기**
- 2-6에서 직접 만든 가중치는 (입력 개수, 1) 형태: (3, 1)
- `nn.Linear`의 가중치는 (출력 개수, 입력 개수) 형태: 같은 모델인데 가로세로가 바뀜

In [9]:
_, _, model = run_and3(2000, 0.1)
weight = model.linear_layer.weight.detach()         # nn.Linear의 가중치
bias = model.linear_layer.bias.detach()
print(f'nn.Linear 가중치 형태: {tuple(weight.shape)}, 편향 형태: {tuple(bias.shape)}')
print(f'2-6의 직접 구현 가중치 형태: {tuple(weight.T.shape)} (nn.Linear 가중치를 전치)')

# 직접 구현 식(X @ weights + bias)에 전치한 가중치를 넣어도 모델의 출력과 같다
manual = 1 / (1 + torch.exp(-(X @ weight.T + bias)))
print(f'직접 구현했을 때와 모델의 출력이 같은가: {torch.allclose(manual, model(X))}')
# torch.allclose() 함수: 두 텐서의 모든 요소가 허용 오차 안에서 같으면 True를 반환
#   - 계산 방식, 데이터 정밀도에 따라 소수점 끝자리가 조금 달라질 수 있음
#   - 이런 경우 ==나 torch.equal() 대신 사용
#   - 절대 차이와 상대 차이를 모두 고려하며, 자세한 사항은 공식 문서 참조

nn.Linear 가중치 형태: (1, 3), 편향 형태: (1,)
2-6의 직접 구현 가중치 형태: (3, 1) (nn.Linear 가중치를 전치)
직접 구현했을 때와 모델의 출력이 같은가: True


**확인**
- `nn.Linear(3, 1)`의 가중치는 (1, 3), 직접 만든 가중치는 (3, 1): 전치 관계
- 전치한 가중치를 직접 구현한 모델에 입력하면 출력이 같음: 같은 계산 결과를 다른 형태로 저장한 것
- 가중치를 직접 꺼내 쓸 때는 `weight[0, 0]`처럼 (출력, 입력) 순서로 접근
- `torch.allclose(a, b)`: 두 텐서의 모든 요소가 허용 오차(상대 1e-5, 절대 1e-8) 안에서 같으면 `True`
    - 부동소수점 계산은 방식이 다르면 끝자리가 조금 달라질 수 있어, 같은지 볼 때는 `==` 대신 이 함수를 사용